# 🧪 Fine-Tuning com LoRA — Modelo Seq2Seq 2
## `google/mt5-small`

Este notebook aplica **LoRA (Low-Rank Adaptation)** ao modelo `mt5-small`,
a versão multilingual do T5 treinada pelo Google em 101 idiomas.

### Por que este modelo?
- O **mT5** (multilingual T5) usa exatamente a mesma arquitetura do T5,
  mas foi pré-treinado no corpus **mC4** cobrindo 101 línguas, incluindo português.
- Diferente do `ptt5` (especializado em PT-BR pela Unicamp), o mT5 **não tem
  especialização em português**: ele conhece o idioma, mas divide recursos
  de representação com 100 outras línguas.
- Isso permite uma **comparação direta** entre os dois Seq2Seq, espelhando
  a comparação entre os modelos causais:

| Par | Modelo especializado PT | Modelo multilingual |
|---|---|---|
| **Causais** | `gpt2-small-portuguese` | `bloom-560m` |
| **Seq2Seq** | `ptt5-base-portuguese-vocab` | `mt5-small` |

### Particularidades do mT5-small
- **Vocabulário**: 250.000 tokens SentencePiece (vs 32.000 do ptt5) — cobre 101 línguas
- **Tamanho**: ~300M parâmetros (encoder + decoder)
- **Sem tarefas supervisionadas no pré-treino**: diferente do T5 original,
  o mT5 foi pré-treinado apenas com span corruption (não viu tarefas de QA).
  Isso torna o fine-tuning supervisionado ainda mais importante.
- **Risco de gibberish**: mT5 com LR alto e poucos dados tende a gerar texto
  incoerente. Usamos LR conservador (1e-4) e label smoothing para mitigar.

### Arquitetura: mT5 vs ptt5
| Aspecto | ptt5-base | mt5-small |
|---|---|---|
| Parâmetros | ~248M | ~300M |
| Vocabulário | 32.000 tokens (PT-BR) | 250.000 tokens (101 línguas) |
| Língua base | PT-BR especializado | Multilingual (101 línguas) |
| Pré-treino | Span corruption + brWaC corpus | Span corruption + mC4 corpus |
| `target_modules` LoRA | `q`, `v` | `q`, `v` (arquitetura idêntica) |


## 📦 1. Instalação das dependências


In [ ]:
!pip install -q transformers datasets peft accelerate sentencepiece
!pip uninstall -y torchao 2>/dev/null || true


## 🔗 2. Montar o Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 📥 3. Importações

Importações idênticas ao Seq2Seq 1 — mT5 usa a mesma classe `AutoModelForSeq2SeqLM`.


In [ ]:
import torch
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType

print(f"PyTorch: {torch.__version__}")
print(f"GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch: 2.11.0+cpu
GPU disponível: False


## 📂 4. Carregar e Preparar o Dataset

### Formato Seq2Seq
Instrução e resposta são mantidas **separadas**, como no Seq2Seq 1:
- **Input (encoder):** `Instruction: <pergunta>`
- **Target (decoder/labels):** `<resposta>`

Mesma `seed=42` do Seq2Seq 1 para garantir o **mesmo split treino/teste**,
tornando as métricas de avaliação comparáveis entre os modelos.


In [ ]:
# ── Configuração ──────────────────────────────────────────────────────────────
DATASET_PATH    = "/content/drive/MyDrive/dataset_gerado_v3 (1).jsonl"
MODEL_SAVE_PATH = "/content/drive/MyDrive/lora_seq2seq_model_2"
MAX_INPUT_LENGTH  = 128
MAX_TARGET_LENGTH = 128
# ──────────────────────────────────────────────────────────────────────────────

dataset = load_dataset("json", data_files=DATASET_PATH)

# Mesma seed do Seq2Seq 1 — split idêntico para comparação justa
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

print(dataset)
print("\nExemplo do dataset:")
print(f"  Instruction: {dataset['train'][0]['Instruction']}")
print(f"  Output:      {dataset['train'][0]['Output']}")

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output'],
        num_rows: 121
    })
    test: Dataset({
        features: ['Instruction', 'Output'],
        num_rows: 31
    })
})

Exemplo do dataset:
  Instruction: Como garantir um desempenho máximo e economia em operações ao usar este equipamento?
  Output:      Para maximizar o desempenho e a economia de operações, é crucial seguir todas as recomendações de segurança, cuidados e manutenção. Isso inclui regularmente ajustar e inspecionar rapidamente os componentes de disco, manter os parafusos afrouxados e monitorar a velocidade do trator conforme solicitado.


## 🤖 5. Carregar o Modelo e o Tokenizador

### Vocabulário mT5: 250.000 tokens
O vocabulário massivo do mT5 tem implicações práticas importantes:

- **Vantagem**: cobertura ampla — palavras técnicas em português raramente
  são fragmentadas em subpalavras.
- **Desvantagem**: o embedding layer é muito maior (~300MB só para os embeddings).
  Isso torna o mT5-small comparável em memória ao ptt5-base apesar do nome 'small'.

### Demonstração de tokenização
Compararemos a tokenização do mT5 com a do ptt5 para o mesmo texto técnico,
evidenciando a diferença entre vocabulário multilingual e especializado em PT.


In [ ]:
model_name = "google/mt5-small"

print(f"Carregando tokenizador: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"pad_token:   '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"eos_token:   '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"Vocabulário: {tokenizer.vocab_size:,} tokens")

print(f"\nCarregando modelo base: {model_name}")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"\nModelo carregado: {model_name}")
print(f"Total de parâmetros: {total_params:,}")
print(f"Memória estimada (float32): {total_params * 4 / 1e9:.2f} GB")

# Comparação de tokenização: mT5 (multilingual) vs ptt5 (PT especializado)
teste_pt = "manutenção e lubrificação das graxeiras do equipamento"
tokens_mt5 = tokenizer.tokenize(teste_pt)
print(f"\nComparação de tokenização:")
print(f"  Texto: '{teste_pt}'")
print(f"  mT5-small  ({len(tokens_mt5)} tokens): {tokens_mt5}")
print(f"  ptt5-base  (referência): ~{len(tokens_mt5)} tokens — execute no notebook S1 para comparar")


Carregando tokenizador: google/mt5-small


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pad_token:   '<pad>' (id=0)
eos_token:   '</s>' (id=1)
Vocabulário: 250,100 tokens

Carregando modelo base: google/mt5-small


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

## 🧪 6. Inferência ANTES do Fine-Tuning (Modelo Base)

Mesma instrução de teste usada em todos os modelos para permitir
comparação direta dos 4 modelos no relatório.

Espera-se que o mT5 base gere texto em português (conhece o idioma),
mas sem conhecimento específico do domínio da Grade Hidráulica GH.
A comparação com o ptt5 base revelará o impacto da especialização em PT.


In [ ]:
def generate_response(model, tokenizer, instruction, max_new_tokens=128):
    """
    Gera resposta com modelo Seq2Seq (mT5).
    Arquitetura idêntica ao ptt5 — mesma função de geração.
    A instrução vai para o encoder; o decoder gera a resposta do zero.
    """
    input_text = f"Instruction: {instruction}"
    device = next(model.parameters()).device

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    resposta = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return resposta


test_instruction = "Qual é a recomendação para transportar o equipamento por longas distâncias?"

print("=" * 60)
print("ANTES DO FINE-TUNING (mt5-small base)")
print("=" * 60)
print(f"Instrução: {test_instruction}")
print(f"Resposta:  {generate_response(base_model, tokenizer, test_instruction)}")


## ✂️ 7. Tokenização para Seq2Seq

### Diferença importante: `as_target_tokenizer()` foi removido no mT5

No Seq2Seq 1 (ptt5), usamos o context manager `tokenizer.as_target_tokenizer()`
para tokenizar os labels. Em versões recentes do HuggingFace Transformers (≥4.26),
esse context manager foi **depreciado** — a forma correta é usar o parâmetro
`text_target` diretamente na chamada do tokenizer:

```python
# Forma antiga (depreciada):
with tokenizer.as_target_tokenizer():
    labels = tokenizer(targets, ...)

# Forma atual (recomendada):
labels = tokenizer(text=inputs, text_target=targets, ...)
```

Isso tokeniza inputs e labels em uma única chamada, o que é mais
eficiente e elimina possíveis inconsistências de configuração entre chamadas.


In [ ]:
def tokenize_seq2seq(examples):
    """
    Tokeniza para Seq2Seq usando a API unificada do tokenizer mT5.

    Diferença do ptt5: usa text_target= em vez de as_target_tokenizer()
    (depreciado em transformers >= 4.26).

    input_ids  → instrução (encoder)
    labels     → resposta  (decoder target)
    """
    inputs  = [f"Instruction: {instr}" for instr in examples["Instruction"]]
    targets = examples["Output"]

    # Tokeniza inputs e targets em uma única chamada
    model_inputs = tokenizer(
        text=inputs,
        text_target=targets,
        max_length=MAX_INPUT_LENGTH,
        max_target_length=MAX_TARGET_LENGTH,
        padding="max_length",
        truncation=True
    )

    return model_inputs


print("Tokenizando dataset para Seq2Seq (mT5)...")
tokenized_datasets = dataset.map(
    tokenize_seq2seq,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Dataset tokenizado:", tokenized_datasets)

# Verificação
sample = tokenized_datasets["train"][0]
print(f"\nVerificação no 1º exemplo:")
print(f"  input_ids (encoder)  — {len(sample['input_ids'])} tokens")
print(f"  Decodificado: {tokenizer.decode(sample['input_ids'], skip_special_tokens=True)[:80]}...")
valid_labels = [l for l in sample['labels'] if l not in [-100, tokenizer.pad_token_id]]
print(f"\n  labels (decoder)     — {len(sample['labels'])} tokens ({len(valid_labels)} ativos)")
print(f"  Decodificado: {tokenizer.decode(valid_labels, skip_special_tokens=True)}")


## 🧩 8. Configurar e Injetar LoRA

### Justificativa dos hiperparâmetros

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `r` | 16 | Mesmo rank do Seq2Seq 1. Mantido igual para comparação controlada entre os dois Seq2Seq. |
| `lora_alpha` | 32 | Escala efetiva α/r = 2.0 — mesma proporção de todos os outros modelos. |
| `target_modules` | `q`, `v` | **Idêntico ao ptt5** — mT5 usa exatamente a mesma arquitetura de atenção T5. Essa igualdade é relevante: isola o efeito do pré-treino (multilingual vs PT-especializado) mantendo a configuração LoRA constante. |
| `lora_dropout` | 0.05 | Mantido igual a todos os outros modelos. |
| `task_type` | `SEQ_2_SEQ_LM` | Igual ao ptt5 — indica ao PEFT que adaptadores devem ser aplicados em encoder e decoder. |

### Confirmação dos módulos-alvo
A arquitetura do mT5 é derivada do T5 original. Os módulos de atenção
têm os mesmos nomes: `q`, `k`, `v`, `o` no encoder e no decoder.
A inspeção abaixo confirma isso antes de configurar o LoRA.


In [ ]:
# Confirma nomes dos módulos de atenção no mT5
print("Camadas de atenção no mt5-small:")
seen = set()
for name, module in base_model.named_modules():
    for key in ["q", "v", "k", "o"]:
        if name.endswith(f".{key}") and name not in seen:
            print(f"  {name} — {type(module).__name__}")
            seen.add(name)
            break


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # Módulos de atenção do mT5 — idênticos ao T5/ptt5
    # Mantidos iguais ao Seq2Seq 1 para comparação controlada
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


## 🧱 9. Data Collator para Seq2Seq

`DataCollatorForSeq2Seq` é idêntico ao usado no Seq2Seq 1:
- Faz padding dinâmico em `input_ids` e `labels` separadamente
- Substitui automaticamente os tokens de padding nos `labels` por `-100`
- Gera `decoder_input_ids` deslocando os labels (teacher forcing)


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)


## ⚙️ 10. Argumentos de Treinamento

### Diferenças em relação ao Seq2Seq 1 (ptt5)

| Parâmetro | Seq2Seq 1 (ptt5) | Seq2Seq 2 (mT5) | Motivo |
|---|---|---|---|
| `learning_rate` | 3e-4 | **1e-4** | mT5 sem fine-tuning supervisionado no pré-treino é mais instável com LR alto — risco de gibberish |
| `label_smoothing_factor` | não definido | **0.1** | Regularização adicional para mitigar overfitting e geração incoerente com dataset pequeno |
| Demais | iguais | iguais | Comparação controlada |

### Por que mT5 com LR alto gera gibberish?
O T5 original foi pré-treinado com tarefas supervisionadas estruturadas
(tradução, sumarização, QA com prefixos). O mT5 foi pré-treinado **apenas
com span corruption** — sem ver nenhuma tarefa de geração condicionada.
Com LR alto e poucos exemplos, os adaptadores LoRA podem distorcer as
representações do encoder antes que o decoder aprenda a usá-las corretamente,
resultando em saídas incoerentes. LR baixo + label smoothing estabiliza esse processo.


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/results_seq2seq_2",
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=1e-4,              # conservador — mT5 sem supervised pretraining
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    label_smoothing_factor=0.1,      # mitiga overfitting e gibberish em mT5
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    report_to="none",
    fp16=torch.cuda.is_available(),
)


## 🏋️ 11. Treinar o Modelo


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Iniciando treinamento...")
train_result = trainer.train()
print("\nTreinamento concluído!")
print(f"Loss final de treino: {train_result.training_loss:.4f}")


## 📊 12. Gráfico de Loss por Step

Compare este gráfico com o do Seq2Seq 1 (ptt5) para avaliar:
- Velocidade de convergência (qual modelo aprende mais rápido)
- Estabilidade do treino (mT5 com LR conservador deve ser mais suave)
- Overfitting (divergência entre train e eval loss)


In [ ]:
log_history = trainer.state.log_history

train_steps, train_losses = [], []
eval_steps, eval_losses   = [], []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label="Train Loss", color="steelblue", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Eval Loss", color="tomato",
         linewidth=2, linestyle="--", marker="o", markersize=5)
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Loss por Step — mt5-small + LoRA\n(Seq2Seq Model 2)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/loss_seq2seq_model_2.png", dpi=150)
plt.show()
print("Gráfico salvo em: /content/drive/MyDrive/loss_seq2seq_model_2.png")


## 💾 13. Salvar o Modelo Ajustado


In [ ]:
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print(f"Modelo salvo em: {MODEL_SAVE_PATH}")

metadata = {
    "modelo_base": "google/mt5-small",
    "tipo": "Seq2Seq (Encoder-Decoder)",
    "arquitetura": "mT5-small (multilingual T5)",
    "multilingual": True,
    "linguas_treinamento": 101,
    "vocabulario": 250000,
    "lora_r": 16,
    "lora_alpha": 32,
    "target_modules": ["q", "v"],
    "task_type": "SEQ_2_SEQ_LM",
    "lora_dropout": 0.05,
    "learning_rate": 1e-4,
    "label_smoothing": 0.1,
    "num_epochs": 10,
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "label_masking": "automático (arquitetura Seq2Seq)",
    "loss_final_treino": round(train_result.training_loss, 4)
}

with open(f"{MODEL_SAVE_PATH}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Metadados salvos.")


## 🔄 14. Carregar o Modelo Salvo para Inferência

Carregamos o modelo fine-tunado do disco para garantir que o salvamento
funcionou corretamente antes de fazer a comparação.


In [ ]:
from peft import PeftModel

base_for_inference = AutoModelForSeq2SeqLM.from_pretrained(
    "google/mt5-small",
    torch_dtype=torch.float32
)

finetuned_model = PeftModel.from_pretrained(base_for_inference, MODEL_SAVE_PATH)
finetuned_model.eval()

finetuned_tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)

print("Modelo fine-tunado carregado com sucesso.")


## 🆚 15. Comparação: Antes vs Depois do Fine-Tuning

Mesma instrução usada em todos os 4 modelos para comparação cruzada.


In [ ]:
test_instruction = "Qual é a recomendação para transportar o equipamento por longas distâncias?"

resposta_base = generate_response(base_model, tokenizer, test_instruction)
resposta_ft   = generate_response(finetuned_model, finetuned_tokenizer, test_instruction)

print("=" * 60)
print(f"Instrução: {test_instruction}")
print("=" * 60)
print(f"\n[ANTES  - mt5-small base]:\n{resposta_base}")
print(f"\n[DEPOIS - mt5-small fine-tunado]:\n{resposta_ft}")
print("\n[REFERÊNCIA do dataset]:")
print("Deve-se utilizar veículo como caminhão, carreta ou prancha.")


## 🔍 16. Testes Adicionais

Mesmas perguntas usadas em todos os modelos para comparação direta no relatório.


In [ ]:
perguntas_teste = [
    "Qual é o rendimento horário trabalhando com um equipamento de 24 discos?",
    "Com que frequência as graxeiras devem ser lubrificadas?",
    "Qual é a velocidade de trabalho recomendada para a Grade Hidráulica GH?",
]

print("=" * 60)
print("TESTES ADICIONAIS — Comparação Base vs Fine-Tunado (mt5-small)")
print("=" * 60)

for pergunta in perguntas_teste:
    r_base = generate_response(base_model, tokenizer, pergunta)
    r_ft   = generate_response(finetuned_model, finetuned_tokenizer, pergunta)
    print(f"\nPergunta: {pergunta}")
    print(f"  Base:        {r_base[:120]}..." if len(r_base) > 120 else f"  Base:        {r_base}")
    print(f"  Fine-tunado: {r_ft[:120]}..."  if len(r_ft)   > 120 else f"  Fine-tunado: {r_ft}")
    print("-" * 60)


## 📋 17. Resumo para o Relatório


In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print("=" * 60)
print("RESUMO — Modelo Seq2Seq 2")
print("=" * 60)
print(f"Modelo base:              google/mt5-small")
print(f"Tipo:                     Seq2Seq (Encoder-Decoder, multilingual)")
print(f"Parâmetros treináveis:    {trainable:,} ({100*trainable/total:.2f}% do total)")
print(f"Parâmetros totais:        {total:,}")
print(f"LoRA rank (r):            16")
print(f"LoRA alpha:               32  (escala efetiva α/r = 2.0)")
print(f"Target modules:           q, v  (idêntico ao ptt5)")
print(f"Task type:                SEQ_2_SEQ_LM")
print(f"LoRA dropout:             0.05")
print(f"Learning rate:            1e-4  (conservador — mT5 sem supervised pretraining)")
print(f"Label smoothing:          0.1")
print(f"Épocas:                   10")
print(f"Max input length:         {MAX_INPUT_LENGTH} tokens")
print(f"Max target length:        {MAX_TARGET_LENGTH} tokens")
print(f"Tokenização labels:       text_target= (API unificada, sem as_target_tokenizer)")
print(f"Label masking:            Automático (arquitetura Seq2Seq)")
print(f"Loss final de treino:     {train_result.training_loss:.4f}")
print(f"Modelo salvo em:          {MODEL_SAVE_PATH}")

print("\n" + "=" * 60)
print("COMPARAÇÃO COMPLETA — 4 MODELOS")
print("=" * 60)
print(f"{'Aspecto':<26} {'C1-GPT2PT':<16} {'C2-BLOOM':<16} {'S1-ptt5':<16} {'S2-mT5'}")
print("-" * 92)
print(f"{'Tipo':<26} {'Causal':<16} {'Causal':<16} {'Seq2Seq':<16} {'Seq2Seq'}")
print(f"{'Parâmetros':<26} {'~124M':<16} {'~560M':<16} {'~248M':<16} {'~300M'}")
print(f"{'Língua base':<26} {'PT-BR':<16} {'Multilingual':<16} {'PT-BR':<16} {'Multilingual'}")
print(f"{'Vocabulário':<26} {'~50k':<16} {'~250k':<16} {'32k PT':<16} {'250k multi'}")
print(f"{'LoRA rank':<26} {'16':<16} {'8':<16} {'16':<16} {'16'}")
print(f"{'Learning rate':<26} {'2e-4':<16} {'1e-4':<16} {'3e-4':<16} {'1e-4'}")
print(f"{'Target modules':<26} {'c_attn,c_proj':<16} {'qkv,dense':<16} {'q,v':<16} {'q,v'}")
print(f"{'Label masking':<26} {'Manual':<16} {'Manual':<16} {'Automático':<16} {'Automático'}")
print(f"{'Label smoothing':<26} {'—':<16} {'—':<16} {'—':<16} {'0.1'}")
